# Databricks Performance Tuning & Best Practices Guide

## Overview
This comprehensive guide covers performance optimization techniques for Databricks, PySpark, Delta Lake, and Spark. All examples are serverless compute compatible.

## Topics Covered
1. **Databricks Fundamentals** - Architecture and compute optimization
2. **PySpark Performance Tuning** - Transformations, actions, and optimization
3. **Delta Lake Best Practices** - ACID transactions, time travel, and optimization
4. **Partitioning Strategies** - Data organization for query performance
5. **Z-Ordering & Data Skipping** - Advanced optimization techniques
6. **Caching Strategies** - When and how to cache data
7. **Broadcast Joins** - Optimizing join operations
8. **Data Skew Handling** - Balancing workloads
9. **File Size Optimization** - Compaction and optimization
10. **Query Optimization** - Predicate pushdown and column pruning
11. **Monitoring & Debugging** - Spark UI insights

---

## 1. Databricks Fundamentals

### Architecture Overview
- **Control Plane**: Manages cluster configuration, notebooks, jobs
- **Data Plane**: Where your Spark clusters run and process data
- **Serverless Compute**: Automatically managed clusters with instant startup

### Serverless Compute Benefits
- ✅ Instant cluster availability (no startup time)
- ✅ Automatic scaling based on workload
- ✅ Optimized Photon engine by default
- ✅ No cluster management overhead
- ✅ Pay only for what you use

### Best Practices for Serverless
1. Let Databricks handle resource allocation
2. Focus on query optimization rather than cluster tuning
3. Use Delta Lake for best performance
4. Leverage adaptive query execution (enabled by default)

In [0]:
# Check current Spark configuration and capabilities
import pyspark
from pyspark.sql import SparkSession

# Get Spark session (already available in Databricks)
spark = SparkSession.builder.getOrCreate()

print("Spark Version:", spark.version)
print("\nKey Configurations:")

# Helper function to safely get config
def get_config(key, default="Not available"):
    try:
        return spark.conf.get(key)
    except:
        return default

print(f"Adaptive Query Execution: {get_config('spark.sql.adaptive.enabled', 'true (default)')}")
print(f"Dynamic Partition Pruning: {get_config('spark.sql.optimizer.dynamicPartitionPruning.enabled', 'true (default)')}")
print(f"Photon Enabled: {get_config('spark.databricks.photon.enabled', 'Auto-managed by serverless')}")

print("\n✅ Serverless compute is ready to use!")

Spark Version: 4.1.0

Key Configurations:
Adaptive Query Execution: true (default)
Dynamic Partition Pruning: true (default)
Photon Enabled: Auto-managed by serverless

✅ Serverless compute is ready to use!


## 2. PySpark Performance Tuning

### Key Concepts

#### Transformations vs Actions
- **Transformations**: Lazy operations that build execution plan (`filter`, `select`, `join`)
- **Actions**: Trigger execution (`count`, `collect`, `show`, `write`)

#### Optimization Principles
1. **Minimize Data Movement** - Filter early, select only needed columns
2. **Avoid Shuffles** - Use broadcast joins when appropriate
3. **Leverage Catalyst Optimizer** - Use DataFrame API over RDD
4. **Predicate Pushdown** - Filter before joins
5. **Column Pruning** - Select only necessary columns

### Common Anti-Patterns
- ❌ Using `collect()` on large datasets
- ❌ Multiple `count()` actions on same DataFrame
- ❌ Not filtering before joins
- ❌ Using UDFs instead of built-in functions
- ❌ Reading entire dataset when only subset needed

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta

# Create sample dataset for demonstration
print("Creating sample sales dataset...")

# Generate sample data
data = [
    (1, 'Product_A', 100.0, '2026-04-01', 'US', 'Electronics'),
    (2, 'Product_B', 150.0, '2026-04-02', 'UK', 'Electronics'),
    (3, 'Product_C', 200.0, '2026-04-03', 'US', 'Clothing'),
    (4, 'Product_A', 120.0, '2026-04-04', 'US', 'Electronics'),
    (5, 'Product_D', 80.0, '2026-04-05', 'CA', 'Clothing'),
    (6, 'Product_B', 160.0, '2026-04-06', 'UK', 'Electronics'),
    (7, 'Product_C', 210.0, '2026-04-07', 'US', 'Clothing'),
    (8, 'Product_A', 110.0, '2026-04-08', 'CA', 'Electronics'),
] * 1000  # Multiply to create larger dataset

df = spark.createDataFrame(data, ['id', 'product', 'amount', 'date', 'country', 'category'])

print(f"✅ Created dataset with {df.count():,} rows")

# GOOD PRACTICE: Column pruning and early filtering
optimized_query = (
    df
    .select('product', 'amount', 'country')  # Select only needed columns EARLY
    .filter(F.col('country') == 'US')  # Filter BEFORE aggregation
    .groupBy('product')
    .agg(F.sum('amount').alias('total_sales'))
    .orderBy(F.desc('total_sales'))
)

print("\n✅ Optimized Query Plan (Column Pruning + Early Filtering):")
optimized_query.explain(mode='simple')

display(optimized_query)

Creating sample sales dataset...
✅ Created dataset with 8,000 rows

✅ Optimized Query Plan (Column Pruning + Early Filtering):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [total_sales#11262 DESC NULLS LAST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7496]
               +- PhotonShuffleExchangeSink rangepartitioning(total_sales#11262 DESC NULLS LAST, 16)
                  +- PhotonGroupingAgg(keys=[product#11264], functions=[finalmerge_sum(merge sum#11271) AS sum(amount)#11269])
                     +- PhotonShuffleExchangeSource
                        +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7490]
                           +- PhotonShuffleExchangeSink hashpartitioning(product#11264, 16)
                              +- PhotonGroupingAgg(keys=[product#11264], functions=[partial_sum(amount#11265) AS sum#11271])
     

product,total_sales
Product_C,410000.0
Product_A,220000.0


In [0]:
# ANTI-PATTERN #1: Multiple actions on same DataFrame
print("❌ ANTI-PATTERN: Multiple count() calls")
print("This triggers separate jobs:")

# Bad - triggers 3 separate Spark jobs
# total_count = df.count()
# us_count = df.filter(F.col('country') == 'US').count()
# uk_count = df.filter(F.col('country') == 'UK').count()

print("\n✅ BETTER APPROACH: Single aggregation")
# Good - single Spark job
counts = df.groupBy('country').count().collect()
for row in counts:
    print(f"{row['country']}: {row['count']:,}")

# ANTI-PATTERN #2: Using collect() on large datasets
print("\n❌ ANTI-PATTERN: collect() on large data")
print("Don't do: all_data = df.collect()  # Brings all data to driver!")

print("\n✅ BETTER: Use limit() or aggregations")
sample = df.limit(10).collect()
print(f"Safely collected {len(sample)} rows for inspection")

# ANTI-PATTERN #3: Not caching expensive computations
print("\n✅ Cache expensive intermediate results (when reused):")
expensive_df = df.groupBy('product', 'category').agg(
    F.sum('amount').alias('total'),
    F.avg('amount').alias('average'),
    F.count('*').alias('count')
)

# Note: For serverless, explicit caching is less critical
# Databricks manages memory automatically
print("Note: Serverless compute handles caching automatically in most cases")

❌ ANTI-PATTERN: Multiple count() calls
This triggers separate jobs:

✅ BETTER APPROACH: Single aggregation
CA: 2,000
US: 4,000
UK: 2,000

❌ ANTI-PATTERN: collect() on large data
Don't do: all_data = df.collect()  # Brings all data to driver!

✅ BETTER: Use limit() or aggregations
Safely collected 10 rows for inspection

✅ Cache expensive intermediate results (when reused):
Note: Serverless compute handles caching automatically in most cases


## 3. Delta Lake Best Practices

### Why Delta Lake?
- **ACID Transactions**: Ensure data consistency
- **Time Travel**: Access historical versions
- **Schema Evolution**: Handle schema changes gracefully
- **MERGE Operations**: Efficient upserts
- **Optimized Performance**: Built-in optimization commands

### Key Features
1. **OPTIMIZE**: Compacts small files
2. **VACUUM**: Removes old files (cleanup)
3. **Z-ORDER**: Co-locates related data
4. **Data Skipping**: Automatic min/max statistics
5. **Liquid Clustering**: Dynamic data organization (new!)

### Best Practices
- ✅ Use Delta format for all production tables
- ✅ Run OPTIMIZE regularly on frequently updated tables
- ✅ Use MERGE for upserts instead of DELETE + INSERT
- ✅ Leverage Z-ORDER for columns used in filters
- ✅ Set appropriate retention periods for VACUUM
- ✅ Use partition columns wisely (not too many!)

In [0]:
# Create a Delta table with optimized settings
from delta.tables import DeltaTable

print("Creating Delta Lake table...")

# Create schema for demo tables if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS demo_perf_tuning")

# Use managed table instead of path
table_name = "demo_perf_tuning.delta_sales"

# Write DataFrame as Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Delta table created: {table_name}")

# Read Delta table
delta_df = spark.table(table_name)

print(f"\nTable contains {delta_df.count():,} rows")
print("\nSample data:")
display(delta_df.limit(5))

Creating Delta Lake table...
✅ Delta table created: demo_perf_tuning.delta_sales

Table contains 8,000 rows

Sample data:


id,product,amount,date,country,category
1,Product_A,100.0,2026-04-01,US,Electronics
2,Product_B,150.0,2026-04-02,UK,Electronics
3,Product_C,200.0,2026-04-03,US,Clothing
4,Product_A,120.0,2026-04-04,US,Electronics
5,Product_D,80.0,2026-04-05,CA,Clothing


In [0]:
# OPTIMIZE: Compact small files for better performance
print("Running OPTIMIZE on Delta table...")

# Using SQL
table_name = "demo_perf_tuning.delta_sales"
spark.sql(f"OPTIMIZE {table_name}")

print("✅ Table optimized - small files compacted")

# Get table details
print("\nTable History:")
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
display(history_df.select('version', 'operation', 'operationMetrics').limit(5))

# Using Python API
print("\n✅ Alternative: Using DeltaTable API")
print("DeltaTable.forName(spark, table_name).optimize().executeCompaction())")

Running OPTIMIZE on Delta table...
✅ Table optimized - small files compacted

Table History:


version,operation,operationMetrics
1,OPTIMIZE,"Map(numRemovedFiles -> 8, numRemovedBytes -> 17152, p25FileSize -> 2215, numDeletionVectorsRemoved -> 0, minFileSize -> 2215, numAddedFiles -> 1, maxFileSize -> 2215, p75FileSize -> 2215, p50FileSize -> 2215, numAddedBytes -> 2215)"
0,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 8000, numOutputBytes -> 17152)"



✅ Alternative: Using DeltaTable API
DeltaTable.forName(spark, table_name).optimize().executeCompaction())


In [0]:
# MERGE: Efficient upsert operation
print("Demonstrating MERGE operation...")

# Create updates/new records
updates_data = [
    (1, 'Product_A', 999.0, '2026-04-12', 'US', 'Electronics'),  # Update existing
    (9999, 'Product_Z', 500.0, '2026-04-12', 'US', 'New_Category'),  # New record
]

updates_df = spark.createDataFrame(updates_data, ['id', 'product', 'amount', 'date', 'country', 'category'])

print("Update dataset:")
display(updates_df)

# Load Delta table
table_name = "demo_perf_tuning.delta_sales"
delta_table = DeltaTable.forName(spark, table_name)

# Perform MERGE (upsert)
print("\nExecuting MERGE operation...")
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("✅ MERGE completed successfully")

# Verify results
result_df = spark.table(table_name)
print(f"\nTable now has {result_df.count():,} rows")
print("\nVerify merged data (id=1 should be updated, id=9999 should be new):")
display(result_df.filter(F.col('id').isin([1, 9999])))

Demonstrating MERGE operation...
Update dataset:


id,product,amount,date,country,category
1,Product_A,999.0,2026-04-12,US,Electronics
9999,Product_Z,500.0,2026-04-12,US,New_Category



Executing MERGE operation...
✅ MERGE completed successfully

Table now has 8,001 rows

Verify merged data (id=1 should be updated, id=9999 should be new):


id,product,amount,date,country,category
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics
1,Product_A,999.0,2026-04-12,US,Electronics


In [0]:
# TIME TRAVEL: Query historical versions
print("Delta Lake Time Travel Examples:")

# Get table history
table_name = "demo_perf_tuning.delta_sales"
history = DeltaTable.forName(spark, table_name).history()
print("\nTable Versions:")
display(history.select('version', 'timestamp', 'operation', 'operationMetrics'))

# Read previous version (before merge)
print("\n✅ Reading version 0 (original data):")
version_0 = spark.read.format("delta").option("versionAsOf", 0).table(table_name)
print(f"Version 0 has {version_0.count():,} rows")

# Read by timestamp
print("\n✅ Can also query by timestamp:")
print("spark.read.format('delta').option('timestampAsOf', '2026-04-12').table(table_name)")

# Restore to previous version
print("\n✅ To restore: RESTORE TABLE table_name TO VERSION AS OF 0")
print("This is useful for recovering from mistakes!")

Delta Lake Time Travel Examples:

Table Versions:


version,timestamp,operation,operationMetrics
3,2026-04-12T17:24:16.000Z,OPTIMIZE,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5893, p25FileSize -> 2293, numDeletionVectorsRemoved -> 1, minFileSize -> 2293, numAddedFiles -> 1, maxFileSize -> 2293, p75FileSize -> 2293, p50FileSize -> 2293, numAddedBytes -> 2293)"
2,2026-04-12T17:24:13.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3678, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1000, executionTimeMs -> 6906, materializeSourceTimeMs -> 389, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2681, numTargetRowsUpdated -> 1000, numOutputRows -> 1001, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3643)"
1,2026-04-12T17:23:53.000Z,OPTIMIZE,"Map(numRemovedFiles -> 8, numRemovedBytes -> 17152, p25FileSize -> 2215, numDeletionVectorsRemoved -> 0, minFileSize -> 2215, numAddedFiles -> 1, maxFileSize -> 2215, p75FileSize -> 2215, p50FileSize -> 2215, numAddedBytes -> 2215)"
0,2026-04-12T17:23:32.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 8000, numOutputBytes -> 17152)"



✅ Reading version 0 (original data):
Version 0 has 8,000 rows

✅ Can also query by timestamp:
spark.read.format('delta').option('timestampAsOf', '2026-04-12').table(table_name)

✅ To restore: RESTORE TABLE table_name TO VERSION AS OF 0
This is useful for recovering from mistakes!


## 4. Partitioning Strategies

### What is Partitioning?
Partitioning divides data into separate directories based on column values, enabling:
- **Partition Pruning**: Skip reading irrelevant data
- **Parallel Processing**: Process partitions independently
- **Faster Queries**: Dramatically reduce data scanned

### Partitioning Best Practices

#### When to Partition
- ✅ Large tables (> 1TB)
- ✅ Queries frequently filter on specific columns
- ✅ Time-based data (date, month, year)
- ✅ Geographic data (country, region)

#### When NOT to Partition
- ❌ Small tables (< 1GB)
- ❌ High cardinality columns (e.g., user_id with millions of values)
- ❌ Columns rarely used in filters
- ❌ Too many partition columns (> 3-4)

### Optimal Partition Size
- **Target**: 1GB - 10GB per partition
- **Too small**: Many tiny files (slow metadata operations)
- **Too large**: No benefit from pruning

### Cardinality Guidelines
- **Good**: 10 - 10,000 partitions
- **Acceptable**: Up to 50,000 partitions
- **Problem**: > 100,000 partitions (avoid!)

In [0]:
# Create partitioned Delta table
print("Creating partitioned Delta table...")

# Generate larger dataset with date column
from datetime import datetime, timedelta
import random

# Create data spanning multiple dates
date_range = [(datetime(2026, 4, 1) + timedelta(days=x)).strftime('%Y-%m-%d') for x in range(10)]
countries = ['US', 'UK', 'CA', 'DE', 'FR']
products = ['Product_A', 'Product_B', 'Product_C', 'Product_D']

large_data = []
for i in range(10000):
    large_data.append((
        i,
        random.choice(products),
        random.uniform(50, 500),
        random.choice(date_range),
        random.choice(countries),
        random.choice(['Electronics', 'Clothing', 'Food'])
    ))

large_df = spark.createDataFrame(large_data, ['id', 'product', 'amount', 'date', 'country', 'category'])

print(f"✅ Created dataset with {large_df.count():,} rows")

# Write with partitioning
partitioned_table = "demo_perf_tuning.partitioned_sales"

large_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("date", "country") \
    .saveAsTable(partitioned_table)

print(f"✅ Partitioned table created: {partitioned_table}")
print("Partitioned by: date, country")

# Show partitioning structure
print("\nPartition structure:")
partitions = spark.sql(f"""
    SELECT date, country, COUNT(*) as record_count
    FROM {partitioned_table}
    GROUP BY date, country
    ORDER BY date, country
    LIMIT 10
""")

display(partitions)

Creating partitioned Delta table...
✅ Created dataset with 10,000 rows
✅ Partitioned table created: demo_perf_tuning.partitioned_sales
Partitioned by: date, country

Partition structure:


date,country,record_count
2026-04-01,CA,169
2026-04-01,DE,177
2026-04-01,FR,205
2026-04-01,UK,212
2026-04-01,US,204
2026-04-02,CA,205
2026-04-02,DE,205
2026-04-02,FR,203
2026-04-02,UK,196
2026-04-02,US,189


In [0]:
# Demonstrate partition pruning
print("Demonstrating Partition Pruning Benefits:")

partitioned_table = "demo_perf_tuning.partitioned_sales"

# Query with partition filter - only reads relevant partitions
print("\n✅ Query with partition pruning (filters on partition columns):")
filtered_query = spark.table(partitioned_table) \
    .filter((F.col('date') == '2026-04-05') & (F.col('country') == 'US'))

print("\nExecution Plan - notice partition filters:")
filtered_query.explain(mode='formatted')

result = filtered_query.count()
print(f"\nReturned {result:,} rows")
print("✅ Only scanned partitions: date=2026-04-05/country=US")

# Compare: Query without partition filter
print("\n❌ Query without partition pruning (no partition column filter):")
print("This would scan ALL partitions:")
no_pruning = spark.table(partitioned_table) \
    .filter(F.col('product') == 'Product_A')

print(f"\nWould scan all {no_pruning.count():,} matching rows across all partitions")

Demonstrating Partition Pruning Benefits:

✅ Query with partition pruning (filters on partition columns):

Execution Plan - notice partition filters:
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet workspace.demo_perf_tuning.partitioned_sales (1)


(1) PhotonScan parquet workspace.demo_perf_tuning.partitioned_sales
Output [6]: [id#13537L, product#13538, amount#13539, category#13542, date#13540, country#13541]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-qtyfr/uc/df103953-d628-4b24-83ea-7a90397e2ee0/cfd7e172-65fc-4fb7-b612-b9a2337a04a5/__unitystorage/catalogs/46fbb487-e485-4072-b196-e4495d92023b/tables/04f7428c-179a-40b7-9116-0c0689879fb5]
PartitionFilters: [isnotnull(date#13540), isnotnull(country#13541), (date#13540 = 2026-04-05), (country#13541 = US)]
ReadSchema: struct<id:bigint,product:string,amount:double,category:string>

(2) PhotonProject
Input [6]: [id#13537L, product#13538, amount#13539, category

In [0]:
# Common partitioning mistakes
print("Common Partitioning Anti-Patterns:\n")

print("❌ ANTI-PATTERN 1: Over-partitioning (too many small partitions)")
print("Example: Partitioning by timestamp (creates thousands of partitions)")
print("DON'T: .partitionBy('timestamp')  # Creates partition per second!")
print("DO: .partitionBy('date')  # Creates partition per day\n")

print("❌ ANTI-PATTERN 2: High cardinality partitioning")
print("DON'T: .partitionBy('user_id')  # Millions of partitions")
print("DO: .partitionBy('date')  # Limited partitions\n")

print("❌ ANTI-PATTERN 3: Too many partition columns")
print("DON'T: .partitionBy('year', 'month', 'day', 'hour', 'country', 'category')")
print("DO: .partitionBy('date', 'country')  # Keep it simple\n")

print("✅ BEST PRACTICE: Partition by columns you frequently filter on")
print("Example: If most queries filter by date and country, use those!\n")

# Show actual file structure
print("Actual partition directory structure:")
print("└── date=2026-04-01/")
print("    ├── country=US/")
print("    │   └── part-00000.parquet")
print("    └── country=UK/")
print("        └── part-00000.parquet")

Common Partitioning Anti-Patterns:

❌ ANTI-PATTERN 1: Over-partitioning (too many small partitions)
Example: Partitioning by timestamp (creates thousands of partitions)
DON'T: .partitionBy('timestamp')  # Creates partition per second!
DO: .partitionBy('date')  # Creates partition per day

❌ ANTI-PATTERN 2: High cardinality partitioning
DON'T: .partitionBy('user_id')  # Millions of partitions
DO: .partitionBy('date')  # Limited partitions

❌ ANTI-PATTERN 3: Too many partition columns
DON'T: .partitionBy('year', 'month', 'day', 'hour', 'country', 'category')
DO: .partitionBy('date', 'country')  # Keep it simple

✅ BEST PRACTICE: Partition by columns you frequently filter on
Example: If most queries filter by date and country, use those!

Actual partition directory structure:
└── date=2026-04-01/
    ├── country=US/
    │   └── part-00000.parquet
    └── country=UK/
        └── part-00000.parquet


## 5. Z-Ordering & Data Skipping

### What is Z-Ordering?
Z-Ordering co-locates related data in the same files using multi-dimensional clustering:
- Improves performance for queries with multiple filter conditions
- Works within partitions (use with partitioning)
- Particularly effective for columns with high cardinality

### When to Use Z-ORDER
- ✅ Columns frequently used in WHERE clauses
- ✅ High cardinality columns (many unique values)
- ✅ Multiple filter columns in queries
- ✅ After partitioning (Z-ORDER within partitions)

### Z-ORDER vs Partitioning

| Aspect | Partitioning | Z-Ordering |
|--------|-------------|------------|
| **Cardinality** | Low (10-10K values) | High (any cardinality) |
| **Columns** | 1-3 columns | Multiple columns |
| **Structure** | Physical directories | File organization |
| **Overhead** | Metadata overhead | Minimal overhead |
| **Use Case** | Date, country, region | User IDs, product IDs |

### Data Skipping
Delta automatically maintains statistics (min/max values) for each file:
- Skips files that don't contain matching data
- Works automatically with Z-ORDERED data
- No configuration needed

In [0]:
# Apply Z-ORDERING to improve query performance
print("Applying Z-ORDER to Delta table...")

partitioned_table = "demo_perf_tuning.partitioned_sales"

# Z-ORDER by columns frequently used in filters
print("\nZ-ORDERing by: product, category")
print("(These are high-cardinality columns often used in WHERE clauses)\n")

spark.sql(f"""
    OPTIMIZE {partitioned_table}
    ZORDER BY (product, category)
""")

print("✅ Z-ORDER completed successfully")
print("\nBenefits:")
print("- Related products and categories co-located in same files")
print("- Data skipping automatically prunes irrelevant files")
print("- Queries filtering on product/category will be faster\n")

# Query that benefits from Z-ORDER
print("Query benefiting from Z-ORDER:")
z_order_query = spark.table(partitioned_table) \
    .filter(
        (F.col('product') == 'Product_A') & 
        (F.col('category') == 'Electronics')
    )

result_count = z_order_query.count()
print(f"\nReturned {result_count:,} rows")
print("✅ Data skipping automatically excluded files without matching data")

display(z_order_query.limit(5))

Applying Z-ORDER to Delta table...

Z-ORDERing by: product, category
(These are high-cardinality columns often used in WHERE clauses)

✅ Z-ORDER completed successfully

Benefits:
- Related products and categories co-located in same files
- Data skipping automatically prunes irrelevant files
- Queries filtering on product/category will be faster

Query benefiting from Z-ORDER:

Returned 859 rows
✅ Data skipping automatically excluded files without matching data


id,product,amount,date,country,category
115,Product_A,404.7797236998922,2026-04-07,US,Electronics
491,Product_A,190.1387246189575,2026-04-07,US,Electronics
2275,Product_A,77.11992791727312,2026-04-07,US,Electronics
2278,Product_A,295.4147248472001,2026-04-07,US,Electronics
3383,Product_A,226.13463096185748,2026-04-07,US,Electronics


In [0]:
# View data skipping statistics
print("Delta Lake Data Skipping Statistics:\n")

print("✅ Delta automatically maintains min/max statistics for each file")
print("This enables data skipping without additional configuration\n")

partitioned_table = "demo_perf_tuning.partitioned_sales"

# Show table statistics
stats = spark.sql(f"""
    DESCRIBE DETAIL {partitioned_table}
""")

print("Table Details:")
display(stats.select('format', 'numFiles', 'sizeInBytes', 'partitionColumns'))

# Show how to check statistics
print("\n✅ To view detailed file statistics:")
print("spark.sql('DESCRIBE EXTENDED table_name')")
print("\nStatistics include:")
print("- Number of files per partition")
print("- Min/max values for each column")
print("- File sizes and locations")
print("- Z-ORDER information")

Delta Lake Data Skipping Statistics:

✅ Delta automatically maintains min/max statistics for each file
This enables data skipping without additional configuration

Table Details:


format,numFiles,sizeInBytes,partitionColumns
delta,50,192179,"List(date, country)"



✅ To view detailed file statistics:
spark.sql('DESCRIBE EXTENDED table_name')

Statistics include:
- Number of files per partition
- Min/max values for each column
- File sizes and locations
- Z-ORDER information


## 6. Broadcast Joins

### What is a Broadcast Join?
- Small table is copied (broadcast) to all executor nodes
- Avoids expensive shuffle operations
- Dramatically faster for joins with small dimension tables

### When to Use Broadcast Joins
- ✅ One table is small (< 100MB by default)
- ✅ Joining fact table with dimension table
- ✅ Lookup tables, reference data
- ✅ Configuration tables

### Broadcast Threshold
- Default: 10MB (`spark.sql.autoBroadcastJoinThreshold`)
- Can be increased for larger dimension tables
- Set to -1 to disable auto-broadcast

### Join Types Supporting Broadcast
- Inner joins
- Left outer joins (broadcast right table)
- Right outer joins (broadcast left table)
- Left semi joins

### Performance Impact
- **Without broadcast**: Shuffle both tables (slow)
- **With broadcast**: No shuffle needed (fast)
- **Savings**: Can be 10-100x faster!

In [0]:
from pyspark.sql.functions import broadcast

# Create a large fact table
print("Creating large fact table (sales transactions)...")

sales_data = []
for i in range(5000):
    sales_data.append((
        i,
        random.randint(1, 20),  # product_id
        random.uniform(10, 1000),
        random.choice(date_range)
    ))

sales_df = spark.createDataFrame(sales_data, ['transaction_id', 'product_id', 'amount', 'date'])
print(f"✅ Sales table: {sales_df.count():,} rows")

# Create a small dimension table
print("\nCreating small dimension table (product catalog)...")

products_data = [
    (1, 'Laptop', 'Electronics'),
    (2, 'Mouse', 'Electronics'),
    (3, 'Keyboard', 'Electronics'),
    (4, 'T-Shirt', 'Clothing'),
    (5, 'Jeans', 'Clothing'),
    (6, 'Apple', 'Food'),
    (7, 'Bread', 'Food'),
]

products_df = spark.createDataFrame(products_data, ['product_id', 'product_name', 'category'])
print(f"✅ Products table: {products_df.count()} rows (small)")

# Join WITHOUT broadcast hint
print("\n❌ Regular join (may trigger shuffle):")
regular_join = sales_df.join(products_df, 'product_id', 'inner')
regular_join.explain(mode='simple')

# Join WITH broadcast hint
print("\n✅ Broadcast join (no shuffle needed):")
broadcast_join = sales_df.join(broadcast(products_df), 'product_id', 'inner')
broadcast_join.explain(mode='simple')

print("\n✅ Broadcast join explicitly ensures no shuffle operation")

# Show results
result = broadcast_join.groupBy('product_name', 'category') \
    .agg(F.sum('amount').alias('total_sales')) \
    .orderBy(F.desc('total_sales'))

display(result)

Creating large fact table (sales transactions)...
✅ Sales table: 5,000 rows

Creating small dimension table (product catalog)...
✅ Products table: 7 rows (small)

❌ Regular join (may trigger shuffle):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [product_id#14039L, transaction_id#14038L, amount#14040, date#14041, product_name#14043, category#14044]
         +- PhotonBroadcastHashJoin [product_id#14039L], [product_id#14042L], Inner, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- LocalTableScan [transaction_id#14038L, product_id#14039L, amount#14040, date#14041]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#10030]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [product_id#14042L, product_name#140

product_name,category,total_sales
Keyboard,Electronics,149033.73890500664
Bread,Food,138606.7045527579
Jeans,Clothing,138273.37154410727
Apple,Food,124857.14415858699
Laptop,Electronics,120811.29679591575
T-Shirt,Clothing,117825.71661432435
Mouse,Electronics,116361.81481821144


In [0]:
# Configure Broadcast Threshold
# Broadcast joins are efficient for small tables (< 10MB by default)

# Helper function for config access
def get_config(key, default):
    try:
        return spark.conf.get(key)
    except:
        return default

# Check current threshold
current_threshold = get_config("spark.sql.autoBroadcastJoinThreshold", "10485760 (default)")
print(f"Current broadcast threshold: {current_threshold}")

# Note: On serverless compute, broadcast settings are managed automatically
# The optimizer will automatically broadcast small tables

# Create example: large and small tables
large_df = spark.range(0, 1000000).withColumn("value", F.col("id") * 2)
small_df = spark.range(0, 100).withColumn("category", F.col("id") % 10)

# Broadcast join (small table automatically broadcast)
result = large_df.join(broadcast(small_df), large_df.id == small_df.id)

print(f"\nBroadcast join completed")
print(f"Result count: {result.count()}")

Current broadcast threshold: 10485760 (default)

Broadcast join completed
Result count: 100


## 7. Handling Data Skew

### What is Data Skew?
Data skew occurs when data is unevenly distributed across partitions:
- Some tasks process much more data than others
- Leads to stragglers (slow tasks that delay completion)
- One executor does all the work while others are idle

### Symptoms of Data Skew
- ❌ One or few tasks take much longer than others
- ❌ Some executors at 100% while others idle
- ❌ Join operations extremely slow
- ❌ OutOfMemory errors on specific tasks

### Common Causes
1. **Skewed Keys**: Popular values (e.g., most users in one country)
2. **Null Values**: Many nulls grouped together
3. **Uneven Partitions**: Poor partitioning strategy
4. **Hot Keys**: Few keys with most data

### Solutions

#### 1. Salting (for joins)
- Add random "salt" to skewed keys
- Distribute hot keys across multiple partitions
- Perform join in two stages

#### 2. Adaptive Query Execution (AQE)
- Enabled by default in Databricks
- Automatically handles skew
- Splits large partitions dynamically

#### 3. Broadcast Join
- If one side is small, broadcast it
- Avoids shuffle entirely

#### 4. Repartitioning
- Redistribute data before processing
- Use `.repartition(n, 'column')`

In [0]:
# Create skewed dataset to demonstrate
print("Creating skewed dataset...")

# Most data in one category (90% in 'Popular')
skewed_data = []
for i in range(9000):
    skewed_data.append((i, 'Popular', random.uniform(10, 100)))

for i in range(9000, 10000):
    skewed_data.append((i, random.choice(['A', 'B', 'C', 'D']), random.uniform(10, 100)))

skewed_df = spark.createDataFrame(skewed_data, ['id', 'category', 'value'])

# Check distribution
print("\nData distribution:")
skew_check = skewed_df.groupBy('category').count().orderBy(F.desc('count'))
display(skew_check)

print("\n❌ Notice: 90% of data in 'Popular' category - this is data skew!")
print("This would cause performance issues in joins and aggregations")

Creating skewed dataset...

Data distribution:


category,count
Popular,9000
C,268
A,255
D,247
B,230



❌ Notice: 90% of data in 'Popular' category - this is data skew!
This would cause performance issues in joins and aggregations


In [0]:
# AQE automatically handles data skew
print("Adaptive Query Execution (AQE) - Automatic Skew Handling:\n")

# Helper function for config access
def get_config(key, default):
    try:
        return spark.conf.get(key)
    except:
        return default

# Check AQE configuration
aqe_enabled = get_config('spark.sql.adaptive.enabled', 'true (default on serverless)')
aqe_skew = get_config('spark.sql.adaptive.skewJoin.enabled', 'true (default)')

print(f"✅ AQE Enabled: {aqe_enabled}")
print(f"✅ AQE Skew Join Optimization: {aqe_skew}")
print("\nNote: On Databricks serverless compute, AQE is enabled by default")
print("AQE automatically detects and handles data skew during execution\n")

# Perform aggregation on skewed data
print("Aggregating skewed data (AQE will optimize automatically):")
agg_result = skewed_df.groupBy('category').agg(
    F.count('*').alias('count'),
    F.avg('value').alias('avg_value')
).orderBy(F.desc('count'))

print("\nAggregation result:")
display(agg_result)

print("\n✅ AQE Benefits:")
print("- Automatically detects skewed partitions during execution")
print("- Splits large partitions into smaller ones")
print("- Dynamically adjusts parallelism")
print("- No manual tuning required on serverless compute!")

Adaptive Query Execution (AQE) - Automatic Skew Handling:

✅ AQE Enabled: true (default on serverless)
✅ AQE Skew Join Optimization: true (default)

Note: On Databricks serverless compute, AQE is enabled by default
AQE automatically detects and handles data skew during execution

Aggregating skewed data (AQE will optimize automatically):

Aggregation result:


category,count,avg_value
Popular,9000,55.44551729248651
C,268,52.68124622444828
A,255,56.044973547353884
D,247,54.59135982033726
B,230,53.343422317784366



✅ AQE Benefits:
- Automatically detects skewed partitions during execution
- Splits large partitions into smaller ones
- Dynamically adjusts parallelism
- No manual tuning required on serverless compute!


In [0]:
# Salting technique for severely skewed joins
print("Salting Technique for Skewed Joins:\n")

# Create another dataset for join
small_lookup = spark.createDataFrame([
    ('Popular', 'Type1'),
    ('A', 'Type2'),
    ('B', 'Type3'),
    ('C', 'Type4'),
    ('D', 'Type5'),
], ['category', 'type'])

print("❌ Problem: Joining skewed table with lookup table")
print("90% of data has category='Popular' - creates hot partition\n")

print("✅ Solution: Add salt to distribute hot keys\n")

# Step 1: Add salt to large table
num_salts = 10
salted_df = skewed_df.withColumn(
    'salt',
    (F.rand() * num_salts).cast('int')
).withColumn(
    'salted_key',
    F.concat(F.col('category'), F.lit('_'), F.col('salt'))
)

print(f"Added {num_salts} salt values to distribute 'Popular' across partitions")

# Step 2: Explode small table with salt
salt_values = list(range(num_salts))
salted_lookup = small_lookup.withColumn(
    'salt',
    F.explode(F.array([F.lit(i) for i in salt_values]))
).withColumn(
    'salted_key',
    F.concat(F.col('category'), F.lit('_'), F.col('salt'))
)

# Step 3: Join on salted key
salted_join = salted_df.join(salted_lookup, 'salted_key', 'inner')

print("\n✅ Salting distributes hot keys across multiple partitions")
print(f"Result count: {salted_join.count():,} rows")

print("\nNote: Salting is an advanced technique for severe skew")
print("Usually AQE handles skew automatically!")

Salting Technique for Skewed Joins:

❌ Problem: Joining skewed table with lookup table
90% of data has category='Popular' - creates hot partition

✅ Solution: Add salt to distribute hot keys

Added 10 salt values to distribute 'Popular' across partitions

✅ Salting distributes hot keys across multiple partitions
Result count: 10,000 rows

Note: Salting is an advanced technique for severe skew
Usually AQE handles skew automatically!


In [0]:
# Repartition to redistribute data evenly
print("Repartitioning Strategy:\n")

print("✅ Repartition by column to distribute data evenly")
print("This ensures data is distributed across partitions based on column values\n")

# Repartition by category column
repartitioned_df = skewed_df.repartition(10, 'category')

print("✅ Repartitioned data by 'category' column into 10 partitions")
print("\nNow data is distributed more evenly across partitions\n")

# Aggregate to show it works
result = repartitioned_df.groupBy('category').agg(
    F.count('*').alias('count'),
    F.avg('value').alias('avg_value')
)

print("Aggregation result after repartitioning:")
display(result)

print("\n✅ Best Practices:")
print("- Repartition by high-cardinality columns")
print("- Use repartition() before expensive operations (joins, aggregations)")
print("- On serverless compute, the optimizer often handles this automatically")
print("\nNote: RDD APIs are not available on serverless compute")
print("Use DataFrame APIs for all operations")

Repartitioning Strategy:

✅ Repartition by column to distribute data evenly
This ensures data is distributed across partitions based on column values

✅ Repartitioned data by 'category' column into 10 partitions

Now data is distributed more evenly across partitions

Aggregation result after repartitioning:


category,count,avg_value
Popular,9000,55.445517292486464
A,255,56.044973547353884
D,247,54.59135982033726
B,230,53.343422317784366
C,268,52.68124622444828



✅ Best Practices:
- Repartition by high-cardinality columns
- Use repartition() before expensive operations (joins, aggregations)
- On serverless compute, the optimizer often handles this automatically

Note: RDD APIs are not available on serverless compute
Use DataFrame APIs for all operations


## 8. File Size Optimization

### The Small Files Problem
- **Issue**: Many small files degrade performance
- **Causes**: Frequent small writes, streaming data, updates
- **Impact**: 
  - Slow metadata operations
  - Inefficient I/O
  - Poor compression
  - Excessive file listing overhead

### Optimal File Size
- **Target**: 128MB - 1GB per file
- **Too small** (< 32MB): Metadata overhead
- **Too large** (> 2GB): Limited parallelism

### Solutions

#### 1. OPTIMIZE Command
```sql
OPTIMIZE table_name
```
- Combines small files into larger ones
- Run regularly on frequently updated tables
- Can be scheduled automatically

#### 2. Auto Optimize (Delta)
- Enable optimized writes automatically
- Auto-compaction after writes

#### 3. Coalesce Before Writing
- Reduce number of output files
- `.coalesce(n)` before `.write()`

#### 4. Repartition for Larger Datasets
- `.repartition(n)` for even distribution
- Better for large datasets

In [0]:
# Analyze file sizes in Delta table
print("Analyzing file sizes...\n")

# Create table with small files (simulating small writes)
small_files_table = "demo_perf_tuning.small_files"

print("Creating table with many small files (anti-pattern)...")

# Write in small batches (creates many small files)
for i in range(5):
    batch = spark.createDataFrame([
        (j, f'value_{j}', random.uniform(1, 100))
        for j in range(i*100, (i+1)*100)
    ], ['id', 'name', 'value'])
    
    batch.write.format("delta").mode("append").saveAsTable(small_files_table)

print("✅ Created table with multiple small files\n")

# Check file details
details = spark.sql(f"DESCRIBE DETAIL {small_files_table}")
print("Table Details:")
display(details.select('numFiles', 'sizeInBytes'))

num_files = details.select('numFiles').collect()[0][0]
total_size = details.select('sizeInBytes').collect()[0][0]
avg_file_size = total_size / num_files / (1024 * 1024) if num_files > 0 else 0

print(f"\n❌ Number of files: {num_files}")
print(f"❌ Average file size: {avg_file_size:.2f} MB")
print(f"\nThis is TOO SMALL! Target: 128MB - 1GB per file")

Analyzing file sizes...

Creating table with many small files (anti-pattern)...
✅ Created table with multiple small files

Table Details:


numFiles,sizeInBytes
5,10625



❌ Number of files: 5
❌ Average file size: 0.00 MB

This is TOO SMALL! Target: 128MB - 1GB per file


In [0]:
# Use OPTIMIZE to compact small files
print("Running OPTIMIZE to compact small files...\n")

small_files_table = "demo_perf_tuning.small_files"

# Before optimization
print("BEFORE OPTIMIZE:")
before_details = spark.sql(f"DESCRIBE DETAIL {small_files_table}")
before_files = before_details.select('numFiles').collect()[0][0]
print(f"Number of files: {before_files}")

# Run OPTIMIZE
optimize_result = spark.sql(f"OPTIMIZE {small_files_table}")
print("\n✅ OPTIMIZE completed")
display(optimize_result)

# After optimization
print("\nAFTER OPTIMIZE:")
after_details = spark.sql(f"DESCRIBE DETAIL {small_files_table}")
after_files = after_details.select('numFiles').collect()[0][0]
after_size = after_details.select('sizeInBytes').collect()[0][0]
avg_size_after = after_size / after_files / (1024 * 1024) if after_files > 0 else 0

print(f"Number of files: {after_files}")
print(f"Average file size: {avg_size_after:.2f} MB")
print(f"\n✅ Reduced from {before_files} files to {after_files} files")
print("This improves query performance significantly!")

Running OPTIMIZE to compact small files...

BEFORE OPTIMIZE:
Number of files: 5

✅ OPTIMIZE completed


path,metrics
,"List(1, 5, List(5948, 5948, 5948.0, 1, 5948), List(2095, 2165, 2125.0, 5, 10625), 0, null, null, 0, 1, 5, 0, true, 0, 0, 1776015127647, 1776015130155, 8, 1, null, List(0, 0), null, 3, 3, 428, 0, null, null)"



AFTER OPTIMIZE:
Number of files: 1
Average file size: 0.01 MB

✅ Reduced from 5 files to 1 files
This improves query performance significantly!


In [0]:
# Enable Auto Optimize for Delta tables
print("Auto Optimize for Delta Tables:\n")

print("✅ Optimized Writes:")
print("Automatically writes larger files (reduces small files)")
print("Enable: .option('delta.autoOptimize.optimizeWrite', 'true')\n")

print("✅ Auto Compaction:")
print("Automatically runs OPTIMIZE after writes")
print("Enable: .option('delta.autoOptimize.autoCompact', 'true')\n")

# Example: Write with auto optimize
auto_optimize_table = "demo_perf_tuning.auto_optimize"

print("Example: Writing with Auto Optimize enabled:\n")

sample_df = spark.range(1000).withColumn('value', F.rand())

sample_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .option("delta.autoOptimize.autoCompact", "true") \
    .saveAsTable(auto_optimize_table)

print("✅ Data written with auto-optimization")

# Check results
details = spark.sql(f"DESCRIBE DETAIL {auto_optimize_table}")
print("\nTable details with auto-optimize:")
display(details.select('numFiles', 'sizeInBytes'))

print("\n✅ Recommendation:")
print("Enable Auto Optimize for frequently updated tables")
print("For batch processing, run OPTIMIZE manually on a schedule")

Auto Optimize for Delta Tables:

✅ Optimized Writes:
Automatically writes larger files (reduces small files)
Enable: .option('delta.autoOptimize.optimizeWrite', 'true')

✅ Auto Compaction:
Automatically runs OPTIMIZE after writes
Enable: .option('delta.autoOptimize.autoCompact', 'true')

Example: Writing with Auto Optimize enabled:

✅ Data written with auto-optimization

Table details with auto-optimize:


numFiles,sizeInBytes
1,9934



✅ Recommendation:
Enable Auto Optimize for frequently updated tables
For batch processing, run OPTIMIZE manually on a schedule


In [0]:
# Use coalesce to control output file count
print("Controlling Output Files with Coalesce:\n")

# Create dataset
large_dataset = spark.range(10000).withColumn('value', F.rand())

print("✅ Created dataset with 10,000 rows\n")

# Write without coalesce
no_coalesce_table = "demo_perf_tuning.no_coalesce"
large_dataset.write.format("delta").mode("overwrite").saveAsTable(no_coalesce_table)

files_without = spark.sql(f"DESCRIBE DETAIL {no_coalesce_table}").select('numFiles').collect()[0][0]
print(f"❌ Without coalesce: {files_without} files created")

# Write with coalesce
with_coalesce_table = "demo_perf_tuning.with_coalesce"
large_dataset.coalesce(2).write.format("delta").mode("overwrite").saveAsTable(with_coalesce_table)

files_with = spark.sql(f"DESCRIBE DETAIL {with_coalesce_table}").select('numFiles').collect()[0][0]
print(f"✅ With coalesce(2): {files_with} files created")

print("\nBest Practices:")
print("✅ Use coalesce() before writing to control file count")
print("✅ Target file size: 128MB - 1GB")
print("✅ Calculate: coalesce(total_size_gb / target_file_size_gb)")
print("\nExample: 10GB dataset, 500MB target = coalesce(20)")
print("\nNote: On serverless compute, the optimizer often handles file sizing automatically")

Controlling Output Files with Coalesce:

✅ Created dataset with 10,000 rows

❌ Without coalesce: 8 files created
✅ With coalesce(2): 2 files created

Best Practices:
✅ Use coalesce() before writing to control file count
✅ Target file size: 128MB - 1GB
✅ Calculate: coalesce(total_size_gb / target_file_size_gb)

Example: 10GB dataset, 500MB target = coalesce(20)

Note: On serverless compute, the optimizer often handles file sizing automatically


## 9. Query Optimization Techniques

### Predicate Pushdown
- **What**: Filters applied as early as possible
- **Benefit**: Reduces data read from storage
- **Automatic**: Catalyst optimizer does this automatically
- **Works with**: Parquet, Delta Lake, ORC

### Column Pruning
- **What**: Only read columns needed for query
- **Benefit**: Reduces I/O significantly
- **Columnar Format**: Essential for Parquet/Delta
- **Best Practice**: Always use SELECT with specific columns

### Query Execution Optimizations

#### 1. Filter Early
```python
# Good: Filter before join
df.filter(...).join(other)

# Bad: Filter after join
df.join(other).filter(...)
```

#### 2. Select Specific Columns
```python
# Good: Select only needed columns
df.select('col1', 'col2')

# Bad: Select all then filter
df.select('*')
```

#### 3. Use SQL Functions
```python
# Good: Built-in functions (optimized)
df.withColumn('new', F.upper('col'))

# Bad: UDFs (slower)
```

#### 4. Leverage Databricks Features
- Photon Engine (automatic in serverless)
- Adaptive Query Execution
- Dynamic File Pruning
- Runtime Filter Pushdown

In [0]:
# Demonstrate query optimization techniques
print("Query Optimization Examples:\n")

# Create test dataset
test_df = spark.range(100000).select(
    F.col('id'),
    (F.col('id') % 100).alias('category'),
    (F.col('id') % 10).alias('subcategory'),
    F.rand().alias('value1'),
    F.rand().alias('value2'),
    F.rand().alias('value3'),
    F.rand().alias('value4'),
    F.rand().alias('value5')
)

print(f"✅ Created test dataset with {test_df.count():,} rows\n")

# BAD: Select all columns, filter late
print("❌ BAD APPROACH:")
print("1. Read all columns (*)")
print("2. Perform expensive operations")
print("3. Filter at the end\n")

bad_query = test_df.select('*') \
    .groupBy('category', 'subcategory') \
    .agg(
        F.avg('value1'),
        F.avg('value2'),
        F.avg('value3'),
        F.avg('value4'),
        F.avg('value5')
    ).filter(F.col('category') < 10)

print("Bad query plan:")
bad_query.explain(mode='simple')

# GOOD: Filter early, select specific columns
print("\n✅ GOOD APPROACH:")
print("1. Filter early (reduce data)")
print("2. Select only needed columns")
print("3. Perform operations on filtered data\n")

good_query = test_df \
    .filter(F.col('category') < 10) \
    .select('category', 'subcategory', 'value1') \
    .groupBy('category', 'subcategory') \
    .agg(F.avg('value1').alias('avg_value'))

print("Good query plan:")
good_query.explain(mode='simple')

print("\n✅ Optimized query processes much less data!")
display(good_query.limit(10))

Query Optimization Examples:

✅ Created test dataset with 100,000 rows

❌ BAD APPROACH:
1. Read all columns (*)
2. Perform expensive operations
3. Filter at the end

Bad query plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[category#16551L, subcategory#16552L], functions=[finalmerge_sum(merge sum#16579) AS sum(value1)#16568, finalmerge_count(merge count#16581L) AS count(1)#16569L, finalmerge_sum(merge sum#16583) AS sum(value2)#16572, finalmerge_sum(merge sum#16585) AS sum(value3)#16570, finalmerge_sum(merge sum#16587) AS sum(value4)#16576, finalmerge_sum(merge sum#16589) AS sum(value5)#16574])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#11917]
               +- PhotonShuffleExchangeSink hashpartitioning(category#16551L, subcategory#16552L, 16)
                  +- PhotonGroupingAgg(keys=[category#16551L, subcatego

category,subcategory,avg_value
3,3,0.5040527539191753
5,5,0.5176008693876822
6,6,0.48947994110915016
9,9,0.47118757896333907
2,2,0.5035703295434456
1,1,0.4993742274969118
0,0,0.4967629113886466
7,7,0.49868484783749584
8,8,0.5064115377882168
4,4,0.5023760510921419


In [0]:
# Demonstrate predicate pushdown with Delta Lake
print("Predicate Pushdown Demonstration:\n")

# Write data to Delta
test_delta_table = "demo_perf_tuning.predicate_pushdown"
test_df.write.format("delta").mode("overwrite").partitionBy("category").saveAsTable(test_delta_table)

print(f"✅ Data written to Delta table: {test_delta_table} (partitioned by category)\n")

# Query with filter - predicate pushdown to storage
query = spark.table(test_delta_table) \
    .filter((F.col('category') == 5) & (F.col('value1') > 0.5)) \
    .select('id', 'category', 'value1')

print("Query with filters on category and value1:")
query.explain(mode='formatted')

print("\n✅ What happened:")
print("1. Partition pruning: Only read category=5 partition")
print("2. Predicate pushdown: Filter value1 > 0.5 at storage layer")
print("3. Column pruning: Only read id, category, value1 columns")
print("\nResult: Dramatically reduced data read from storage!")

result_count = query.count()
print(f"\nReturned {result_count:,} rows (out of 100,000 total)")

Predicate Pushdown Demonstration:

✅ Data written to Delta table: demo_perf_tuning.predicate_pushdown (partitioned by category)

Query with filters on category and value1:
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet workspace.demo_perf_tuning.predicate_pushdown (1)


(1) PhotonScan parquet workspace.demo_perf_tuning.predicate_pushdown
Output [3]: [id#16963L, value1#16966, category#16964L]
DictionaryFilters: [(value1#16966 > 0.5)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-qtyfr/uc/df103953-d628-4b24-83ea-7a90397e2ee0/cfd7e172-65fc-4fb7-b612-b9a2337a04a5/__unitystorage/catalogs/46fbb487-e485-4072-b196-e4495d92023b/tables/52d398d4-62f8-4cc5-b7ec-91991832ff64]
PartitionFilters: [isnotnull(category#16964L), (category#16964L = 5)]
ReadSchema: struct<id:bigint,value1:double>
RequiredDataFilters: [isnotnull(value1#16966), (value1#16966 > 0.5)]

(2) PhotonProject
Input [3]: [id#16963L, value1#16966, categor

In [0]:
# Compare built-in functions vs UDFs
print("Built-in Functions vs UDFs Performance:\n")

# Sample dataset
sample = spark.range(10000).withColumn('text', F.lit('HELLO WORLD'))

print("✅ GOOD: Built-in function (optimized by Catalyst)")
print("result = df.withColumn('lower_text', F.lower('text'))")
print("- Executed in native code (Java/Scala)")
print("- Optimized by Catalyst")
print("- Can be pushed down to storage\n")

good_result = sample.withColumn('lower_text', F.lower('text'))
print("Built-in function plan:")
good_result.explain(mode='simple')

print("\n❌ AVOID: Python UDF (slower)")
print("@udf")
print("def lower_udf(text):")
print("    return text.lower()")
print("result = df.withColumn('lower_text', lower_udf('text'))")
print("- Serializes data to Python")
print("- No optimization possible")
print("- Cannot be pushed down\n")

print("✅ Best Practices:")
print("1. Always use built-in Spark functions (pyspark.sql.functions)")
print("2. Use Pandas UDFs if UDF is absolutely necessary")
print("3. Avoid row-by-row Python UDFs")
print("4. Consider moving complex logic to Delta Live Tables")

Built-in Functions vs UDFs Performance:

✅ GOOD: Built-in function (optimized by Catalyst)
result = df.withColumn('lower_text', F.lower('text'))
- Executed in native code (Java/Scala)
- Optimized by Catalyst
- Can be pushed down to storage

Built-in function plan:
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [id#17074L, HELLO WORLD AS text#17076, hello world AS lower_text#17078]
      +- PhotonRange Range (0, 10000, step=1, splits=8)


== Photon Explanation ==
The query is fully supported by Photon.

❌ AVOID: Python UDF (slower)
@udf
def lower_udf(text):
    return text.lower()
result = df.withColumn('lower_text', lower_udf('text'))
- Serializes data to Python
- No optimization possible
- Cannot be pushed down

✅ Best Practices:
1. Always use built-in Spark functions (pyspark.sql.functions)
2. Use Pandas UDFs if UDF is absolutely necessary
3. Avoid row-by-row Python UDFs
4. Consider moving complex logic to Delta Live Tables


## 10. Monitoring & Debugging with Spark UI

### Accessing Spark UI
- **Databricks**: Click "View" → "Spark UI" from notebook or cluster
- **Key Tabs**: Jobs, Stages, Storage, Environment, Executors, SQL

### What to Monitor

#### 1. Jobs Tab
- Duration of each job
- Number of stages
- Failed tasks

#### 2. Stages Tab
- **Task duration**: Look for stragglers (slow tasks)
- **Data skew**: Uneven task processing times
- **Shuffle read/write**: Large shuffles are expensive
- **GC time**: High GC indicates memory pressure

#### 3. SQL Tab (Most Important for DataFrames)
- **Query execution plan**: Physical and logical plans
- **Scan metrics**: How much data was read
- **Shuffle metrics**: Data movement between stages
- **Time breakdown**: Where time is spent

#### 4. Storage Tab
- **Cached RDDs**: What's in memory
- **Memory usage**: How much memory used

### Key Metrics to Watch

| Metric | Good | Problem |
|--------|------|--------|
| **Task Duration** | Even distribution | Few tasks take 10x longer (skew) |
| **Shuffle Read** | < 1GB per task | > 5GB per task |
| **GC Time** | < 10% of task time | > 30% of task time |
| **Spill (Disk)** | 0 GB | Any spilling indicates memory issues |
| **Tasks** | All succeed quickly | Many retries or failures |

### Common Issues

#### Data Skew
- **Symptom**: Few tasks take much longer
- **Solution**: Salting, AQE, repartition

#### Memory Issues
- **Symptom**: Spill to disk, OutOfMemory errors
- **Solution**: Increase executor memory, reduce partition size

#### Shuffle Issues
- **Symptom**: Large shuffle read/write
- **Solution**: Broadcast joins, better partitioning, filter early

In [0]:
# Generate workload to demonstrate Spark UI insights
print("Generating sample workload for Spark UI analysis...\n")

# Create diverse dataset
monitoring_df = spark.range(50000).select(
    F.col('id'),
    (F.col('id') % 100).alias('partition_key'),
    F.rand().alias('value1'),
    F.rand().alias('value2')
)

print("✅ Created dataset for monitoring demo\n")

# Perform various operations
print("Executing different types of operations:")

# 1. Simple aggregation
print("\n1. Simple aggregation (check Jobs tab)")
agg_result = monitoring_df.groupBy('partition_key').agg(
    F.count('*').alias('count'),
    F.avg('value1').alias('avg_value1')
)
print(f"   Result: {agg_result.count()} rows")

# 2. Join operation (will show shuffle)
print("\n2. Join operation (check SQL tab for shuffle metrics)")
df1 = monitoring_df.sample(0.1)
df2 = monitoring_df.sample(0.1).withColumnRenamed('value1', 'other_value')
join_result = df1.join(df2, 'id', 'inner')
print(f"   Result: {join_result.count()} rows")

# 3. Multiple transformations
print("\n3. Complex transformation chain (check Stages tab)")
complex_result = monitoring_df \
    .filter(F.col('value1') > 0.5) \
    .withColumn('value_squared', F.col('value1') * F.col('value1')) \
    .groupBy('partition_key') \
    .agg(
        F.sum('value_squared').alias('sum_squared'),
        F.max('value2').alias('max_value2')
    ) \
    .orderBy(F.desc('sum_squared'))

print(f"   Result: {complex_result.count()} rows")
display(complex_result.limit(10))

print("\n✅ Workload complete! Now check the Spark UI:")
print("   → View → Spark UI")
print("   → Click on 'SQL' tab to see query plans and metrics")
print("   → Click on 'Stages' tab to see task-level details")

Generating sample workload for Spark UI analysis...

✅ Created dataset for monitoring demo

Executing different types of operations:

1. Simple aggregation (check Jobs tab)
   Result: 100 rows

2. Join operation (check SQL tab for shuffle metrics)
   Result: 536 rows

3. Complex transformation chain (check Stages tab)
   Result: 100 rows


partition_key,sum_squared,max_value2
38,167.78064307106155,0.9967177513048235
56,160.51605247559496,0.9981380739138085
16,157.9108457142616,0.9990027123947055
74,157.71237303858018,0.9986027530018595
22,156.3470898689019,0.9989381430885944
24,155.93290060688636,0.9998735501018486
41,155.89369723090832,0.9870394158087914
36,155.6034111304838,0.998614685834055
19,154.71909869870984,0.995464621479204
27,154.4961525238969,0.9957593012872041



✅ Workload complete! Now check the Spark UI:
   → View → Spark UI
   → Click on 'SQL' tab to see query plans and metrics
   → Click on 'Stages' tab to see task-level details


In [0]:
# Guide to analyzing Spark UI
print("Spark UI Analysis Guide:\n")

print("✅ HOW TO USE SPARK UI:\n")

print("1. SQL TAB (Best for DataFrame/SQL queries):")
print("   - Shows all SQL queries executed")
print("   - Click query to see execution plan")
print("   - Look for:")
print("     * Scan: How much data read")
print("     * Exchange: Shuffle operations (expensive!)")
print("     * WholeStageCodegen: Good (optimized execution)")
print("     * Duration: Where time is spent\n")

print("2. JOBS TAB:")
print("   - One job per action (count, show, write)")
print("   - Look for:")
print("     * Duration: Total job time")
print("     * Stages: How many stages (fewer = better)")
print("     * Failed tasks: Indicates problems\n")

print("3. STAGES TAB (For detailed task analysis):")
print("   - Shows individual tasks within stages")
print("   - Look for:")
print("     * Task duration distribution (histogram)")
print("     * Skew: Some tasks much slower")
print("     * Shuffle read/write: Data movement")
print("     * GC time: Memory pressure\n")

print("4. EXECUTORS TAB:")
print("   - Resource utilization")
print("   - Memory usage")
print("   - Disk spill (bad if > 0)\n")

print("✅ RED FLAGS TO WATCH FOR:\n")
print("❌ Large shuffle size (> 1GB per task)")
print("❌ High GC time (> 30% of task time)")
print("❌ Data skew (few tasks take 10x longer)")
print("❌ Disk spill (any amount)")
print("❌ Many failed/retried tasks\n")

print("✅ GOOD SIGNS:\n")
print("✓ WholeStageCodegen present")
print("✓ Even task duration distribution")
print("✓ Low shuffle sizes")
print("✓ No disk spill")
print("✓ Low GC time (< 10%)")

Spark UI Analysis Guide:

✅ HOW TO USE SPARK UI:

1. SQL TAB (Best for DataFrame/SQL queries):
   - Shows all SQL queries executed
   - Click query to see execution plan
   - Look for:
     * Scan: How much data read
     * Exchange: Shuffle operations (expensive!)
     * WholeStageCodegen: Good (optimized execution)
     * Duration: Where time is spent

2. JOBS TAB:
   - One job per action (count, show, write)
   - Look for:
     * Duration: Total job time
     * Stages: How many stages (fewer = better)
     * Failed tasks: Indicates problems

3. STAGES TAB (For detailed task analysis):
   - Shows individual tasks within stages
   - Look for:
     * Task duration distribution (histogram)
     * Skew: Some tasks much slower
     * Shuffle read/write: Data movement
     * GC time: Memory pressure

4. EXECUTORS TAB:
   - Resource utilization
   - Memory usage
   - Disk spill (bad if > 0)

✅ RED FLAGS TO WATCH FOR:

❌ Large shuffle size (> 1GB per task)
❌ High GC time (> 30% of task time)

## 11. Performance Tuning Checklist

### ✅ Data Format
- [ ] Use Delta Lake for all production tables
- [ ] Run OPTIMIZE regularly
- [ ] Use Z-ORDER for frequently filtered columns
- [ ] Enable Auto Optimize for frequently updated tables

### ✅ Query Optimization
- [ ] Filter as early as possible
- [ ] Select only needed columns
- [ ] Use built-in functions instead of UDFs
- [ ] Leverage predicate pushdown
- [ ] Check query plans in Spark UI

### ✅ Joins
- [ ] Broadcast small tables (< 100MB)
- [ ] Filter before joining
- [ ] Use appropriate join types
- [ ] Watch for data skew in joins

### ✅ Partitioning
- [ ] Partition large tables by frequently filtered columns
- [ ] Target 1-10GB per partition
- [ ] Avoid high cardinality partitions
- [ ] Use 1-3 partition columns maximum

### ✅ File Management
- [ ] Target 128MB - 1GB per file
- [ ] Compact small files with OPTIMIZE
- [ ] Use coalesce before writing
- [ ] Monitor file counts

### ✅ Memory & Resources
- [ ] Let serverless compute auto-scale
- [ ] Don't use .cache() unless necessary
- [ ] Avoid collect() on large datasets
- [ ] Monitor Spark UI for memory issues

### ✅ Data Skew
- [ ] Enable AQE (automatic in serverless)
- [ ] Use salting for severely skewed joins
- [ ] Repartition if needed
- [ ] Monitor task duration distribution

### ✅ Monitoring
- [ ] Check Spark UI after query execution
- [ ] Look for shuffle operations
- [ ] Identify slow tasks (stragglers)
- [ ] Monitor GC time and memory
- [ ] Review execution plans

## 12. Quick Reference Commands

### Delta Lake Operations

```python
# Read Delta table
df = spark.read.format("delta").load("/path/to/table")

# Write Delta table
df.write.format("delta").mode("overwrite").save("/path/to/table")

# Optimize (compact files)
spark.sql("OPTIMIZE table_name")
spark.sql("OPTIMIZE table_name ZORDER BY (col1, col2)")

# Vacuum (remove old files)
spark.sql("VACUUM table_name RETAIN 168 HOURS")  # 7 days

# Time Travel
df = spark.read.format("delta").option("versionAsOf", 0).load("/path")
df = spark.read.format("delta").option("timestampAsOf", "2026-04-01").load("/path")

# MERGE (upsert)
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/path")
delta_table.alias("t").merge(
    source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

### Performance Tuning

```python
# Broadcast join
from pyspark.sql.functions import broadcast
df1.join(broadcast(df2), "key")

# Repartition
df.repartition(10, "column")

# Coalesce (reduce partitions without shuffle)
df.coalesce(5)

# Cache (use sparingly)
df.cache()
df.unpersist()

# Explain query plan
df.explain(mode="formatted")
```

### Configuration

```python
# Check configuration
spark.conf.get("spark.sql.adaptive.enabled")

# Set configuration  
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "100MB")
```

### Monitoring

```python
# Get number of partitions
df.rdd.getNumPartitions()

# Check table details
spark.sql("DESCRIBE DETAIL table_name")
spark.sql("DESCRIBE HISTORY table_name")

# Show execution plan
df.explain()
df.explain(mode="cost")
df.explain(mode="formatted")
```

## Summary & Next Steps

### Key Takeaways

1. **Use Delta Lake**: ACID transactions, time travel, and built-in optimization
2. **Optimize Queries**: Filter early, select specific columns, use built-in functions
3. **Manage Files**: Run OPTIMIZE regularly, target 128MB-1GB per file
4. **Smart Partitioning**: Use for large tables, avoid high cardinality
5. **Leverage Broadcast**: For joins with small dimension tables
6. **Monitor Performance**: Use Spark UI to identify bottlenecks
7. **Trust Serverless**: Let Databricks handle resource management with AQE

### Performance Hierarchy (Biggest Impact First)

1. ⭐⭐⭐ **Data Format** (Delta Lake) - 10-100x improvement
2. ⭐⭐⭐ **Query Design** (filter early, column pruning) - 5-50x improvement
3. ⭐⭐ **File Size** (OPTIMIZE) - 2-10x improvement
4. ⭐⭐ **Partitioning** (right strategy) - 2-10x improvement
5. ⭐⭐ **Join Strategy** (broadcast vs shuffle) - 2-10x improvement
6. ⭐ **Z-ORDER** (for specific queries) - 1.5-3x improvement

### Serverless Compute Advantages

✅ **Automatic Optimization**
- Photon engine enabled by default
- Adaptive Query Execution (AQE) configured optimally
- Auto-scaling based on workload
- No cluster management needed

✅ **Focus on Code, Not Infrastructure**
- Write efficient queries
- Use Delta Lake best practices
- Monitor with Spark UI
- Let Databricks handle the rest

### Next Steps

1. **Run the examples** in this notebook to see optimizations in action
2. **Apply techniques** to your actual datasets
3. **Monitor results** using Spark UI
4. **Iterate** and refine based on metrics
5. **Schedule OPTIMIZE** for production tables

### Additional Resources

- [Databricks Documentation](https://docs.databricks.com)
- [Delta Lake Documentation](https://docs.delta.io)
- [Spark Performance Tuning](https://spark.apache.org/docs/latest/tuning.html)
- [Databricks Academy](https://academy.databricks.com)

---
